In [2]:
import time
import requests
import pandas as pd
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DATA_DIR = Path.cwd() / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'data'

df = pd.read_csv(DATA_DIR / 'final_data.csv', low_memory=False)
df['hs_id'] = df['hs_id'].astype('string').str.strip()
df['cycle'] = pd.to_numeric(df['cycle'], errors='coerce').astype('Int64')
print('final_data shape:', df.shape)
df.head(2)

final_data shape: (1048575, 44)


,college_id,cycle,school_type,hs_id,col_name,col_city,col_st,col_zip,col_type,col_ctyname,...,hs_magnet,hs_enrollment,hs_pct_free_or_reduced_price_lunch,hs_students_per_teacher,hs_priv_level,hs_priv_pinst,hs_priv_pcnty,hs_priv_pstabb,hs_school_level,hs_highest_grade_offered
0,45896401,2022,public,450201000242,STRAYER UNIVERSITY - CHARLESTON CAMPUS,NORTH CHARLESTON,SC,29418.0,3.0,CHARLESTON,...,NaN,850.0,0.757647,14.655172,NaN,NaN,NaN,NaN,2.0,8.0
1,101189,2023,private,A1900014,FAULKNER UNIVERSITY,MONTGOMERY,AL,36109.0,2.0,MONTGOMERY,...,NaN,NaN,NaN,NaN,3.0,EASTWOOD CHRISTIAN SCHOOL,101.0,AL,NaN,NaN


In [3]:
# Fill geo columns for all private rows from PSS 2019-20 (used for all cycles)
pss_1920 = pd.read_csv(DATA_DIR / 'pss_2019-20.csv', low_memory=False,
                       usecols=['PPIN', 'PSTABB', 'PZIP', 'PCNTNM', 'PCNTY', 'PCITY', 'LATITUDE20', 'LONGITUDE20'])
pss_1920['PPIN'] = pss_1920['PPIN'].astype('string').str.strip()
pss_1920 = pss_1920.rename(columns={'LATITUDE20': 'LATITUDE', 'LONGITUDE20': 'LONGITUDE'})
pss_1920_dedup = pss_1920.drop_duplicates(subset=['PPIN'])

# Column mapping: PSS -> final_data
geo_map = {
    'PSTABB':    'hs_state',
    'PZIP':      'hs_zip',
    'PCITY':     'hs_city',
    'PCNTNM':    'hs_ctyname',
    'PCNTY':     'hs_cty_fips',
    'LATITUDE':  'hs_lat',
    'LONGITUDE': 'hs_long',
}

priv_mask = df['school_type'].astype('string') == 'private'

merged = (
    df.loc[priv_mask, ['hs_id']]
      .merge(pss_1920_dedup[['PPIN'] + list(geo_map.keys())], left_on='hs_id', right_on='PPIN', how='left')
      .set_index(df.loc[priv_mask].index)
)

for pss_col, hs_col in geo_map.items():
    missing = priv_mask & df[hs_col].isna()
    df.loc[missing, hs_col] = merged.loc[missing, pss_col]
    print(f'  {hs_col}: filled {missing.sum():,} rows')

print('\nMissing % for private rows after fill:')
print((df.loc[priv_mask, list(geo_map.values())].isna().mean() * 100).round(2))

  hs_state: filled 137,449 rows
  hs_zip: filled 137,449 rows
  hs_city: filled 137,449 rows
  hs_ctyname: filled 137,449 rows
  hs_cty_fips: filled 137,449 rows
  hs_lat: filled 137,449 rows
  hs_long: filled 137,449 rows

Missing % for private rows after fill:
hs_state       0.0
hs_zip         0.0
hs_city        0.0
hs_ctyname     0.0
hs_cty_fips    0.0
hs_lat         0.0
hs_long        0.0
dtype: float64


In [4]:
# Fetch school_name from CCD directory API for public schools (all years)
session = requests.Session()
retries = Retry(
    total=8, connect=8, read=8, status=8,
    backoff_factor=0.75,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(['GET']),
    respect_retry_after_header=True,
)
session.mount('https://', HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10))
session.headers.update({'User-Agent': 'c2i-ccd-directory/1.0'})

def fetch_json(url):
    for attempt in range(6):
        try:
            resp = session.get(url, timeout=(15, 120))
            resp.raise_for_status()
            return resp.json()
        except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError):
            if attempt == 5:
                raise
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError('Unreachable')

years = sorted(df['cycle'].dropna().unique().astype(int).tolist())
print('Fetching CCD school_name for years:', years)

year_dfs = []
for year in years:
    url = f'https://educationdata.urban.org/api/v1/schools/ccd/directory/{year}/'
    rows = []
    while url:
        data = fetch_json(url)
        rows.extend(data.get('results', []))
        url = data.get('next')
        time.sleep(0.05)
    df_year = pd.DataFrame(rows)[['ncessch', 'school_name']].copy()
    df_year['cycle'] = year
    year_dfs.append(df_year)
    print(f'  {year}: {len(rows):,} rows')

ccd_names = pd.concat(year_dfs, ignore_index=True)
ccd_names['ncessch'] = ccd_names['ncessch'].astype('string').str.strip()
ccd_names['cycle'] = ccd_names['cycle'].astype('Int64')
ccd_names = ccd_names.drop_duplicates(subset=['ncessch', 'cycle'])
print('ccd_names shape:', ccd_names.shape)

Fetching CCD school_name for years: [2019, 2020, 2021, 2022, 2023]
  2019: 101,688 rows
  2020: 101,662 rows
  2021: 102,130 rows
  2022: 102,268 rows
  2023: 102,274 rows
ccd_names shape: (510022, 3)


In [5]:
# Merge CCD names and build unified hs_name column
df = df.merge(
    ccd_names.rename(columns={'ncessch': 'hs_id', 'school_name': '_ccd_name'}),
    on=['hs_id', 'cycle'],
    how='left'
)

# Public: CCD school_name; Private: hs_priv_pinst
df['hs_name'] = df['_ccd_name'].where(
    df['school_type'].astype('string') == 'public',
    other=df['hs_priv_pinst']
)
df.drop(columns=['_ccd_name'], inplace=True)

for stype in ['public', 'private']:
    mask = df['school_type'].astype('string') == stype
    pct = df.loc[mask, 'hs_name'].notna().mean() * 100
    print(f'hs_name fill rate ({stype}): {pct:.2f}%')

hs_name fill rate (public): 98.67%
hs_name fill rate (private): 100.00%


In [6]:
# Drop redundant private-specific columns now unified into shared cols or hs_name
# hs_priv_pstabb -> hs_state, hs_priv_pcnty -> hs_cty_fips, hs_priv_pinst -> hs_name
drop_cols = ['hs_priv_pinst', 'hs_priv_pcnty', 'hs_priv_pstabb']
df.drop(columns=drop_cols, inplace=True, errors='ignore')
print('Dropped:', drop_cols)

# Position hs_name right after hs_id
cols = df.columns.tolist()
cols.remove('hs_name')
cols.insert(cols.index('hs_id') + 1, 'hs_name')
df = df[cols]

print('Final shape:', df.shape)
print('Final columns:', df.columns.tolist())

Dropped: ['hs_priv_pinst', 'hs_priv_pcnty', 'hs_priv_pstabb']
Final shape: (1048575, 42)
Final columns: ['college_id', 'cycle', 'school_type', 'hs_id', 'hs_name', 'col_name', 'col_city', 'col_st', 'col_zip', 'col_type', 'col_ctyname', 'col_ctyfips', 'col_shparea', 'col_shplength', 'hs_city', 'hs_state', 'hs_zip', 'hs_ctyname', 'hs_cty_fips', 'hs_lat', 'hs_long', 'col_inst_control', 'col_inst_size', 'col_endow_total', 'col_endow_per_fte', 'hs_pct_white', 'hs_pct_black', 'hs_pct_hispanic', 'hs_pct_asian', 'hs_pct_aian', 'hs_pct_nhpi', 'hs_pct_two_or_more', 'hs_title_i_status', 'hs_urban_centric_locale', 'hs_charter', 'hs_magnet', 'hs_enrollment', 'hs_pct_free_or_reduced_price_lunch', 'hs_students_per_teacher', 'hs_priv_level', 'hs_school_level', 'hs_highest_grade_offered']


In [7]:
# Recode hs_school_level to unified string labels for both public and private
# Public codes: 0=PreK, 1=Primary, 2=Middle, 3=High, 4=Other, 5=Ungraded, 6=Adult Ed, 7=Secondary
# Private codes (hs_priv_level): 1=Elementary, 2=Secondary, 3=Combined
pub_level_map = {
    0:  'pre_k',
    1:  'elementary',
    2:  'middle',
    3:  'high',
    4:  'other',
    5:  'ungraded',
    6:  'adult_ed',
    7:  'secondary',
    -1: None,
    -2: None,
    -3: None,
}

priv_level_map = {
    1: 'elementary',
    2: 'secondary',
    3: 'combined',
}

pub_mask  = df['school_type'].astype('string') == 'public'
priv_mask = df['school_type'].astype('string') == 'private'

pub_mapped  = pd.to_numeric(df['hs_school_level'], errors='coerce').astype('Int64').map(pub_level_map)
priv_mapped = pd.to_numeric(df['hs_priv_level'],   errors='coerce').astype('Int64').map(priv_level_map)

df['hs_school_level'] = pub_mapped.where(pub_mask).combine_first(priv_mapped.where(priv_mask))

df.drop(columns=['hs_priv_level'], inplace=True, errors='ignore')

print('hs_school_level value counts:')
print(df['hs_school_level'].value_counts(dropna=False))

hs_school_level value counts:
hs_school_level
elementary    469054
high          272565
middle        143722
combined       65184
secondary      34674
other          29169
NaN            23894
pre_k           8739
ungraded        1383
adult_ed         191
Name: count, dtype: int64


In [8]:
# Recode hs_highest_grade_offered for private schools
# Map PSS HIGR2020 codes onto the public numeric grade scale:
#   Public: -1=Pre-K, 0=K, 1-12=grades 1-12, 14=Adult Ed, 15=Ungraded
higr_map = {
    1:  15,  # All Ungraded -> 15 (Ungraded)
    2:  -1,  # Highest grade is Pre-K
    3:   0,  # Kindergarten
    4:   0,  # Transitional K -> K (approximate)
    5:   1,  # Transitional 1st -> grade 1 (approximate)
    6:   1,  # 1st grade
    7:   2,
    8:   3,
    9:   4,
    10:  5,
    11:  6,
    12:  7,
    13:  8,
    14:  9,
    15: 10,
    16: 11,
    17: 12,
}

pss_higr = pd.read_csv(DATA_DIR / 'pss_2019-20.csv', low_memory=False, usecols=['PPIN', 'HIGR2020'])
pss_higr['PPIN'] = pss_higr['PPIN'].astype('string').str.strip()
pss_higr = pss_higr.drop_duplicates(subset=['PPIN'])
pss_higr['higr_mapped'] = pd.to_numeric(pss_higr['HIGR2020'], errors='coerce').map(higr_map)

priv_mask = df['school_type'].astype('string') == 'private'

merged_higr = (
    df.loc[priv_mask, ['hs_id']]
      .merge(pss_higr[['PPIN', 'higr_mapped']], left_on='hs_id', right_on='PPIN', how='left')
      .set_index(df.loc[priv_mask].index)
)

df.loc[priv_mask, 'hs_highest_grade_offered'] = merged_higr['higr_mapped']

print('hs_highest_grade_offered value counts (private):')
print(df.loc[priv_mask, 'hs_highest_grade_offered'].value_counts(dropna=False).sort_index())
print('\nhs_highest_grade_offered value counts (public):')
print(df.loc[~priv_mask, 'hs_highest_grade_offered'].value_counts(dropna=False).sort_index())

hs_highest_grade_offered value counts (private):
hs_highest_grade_offered
-1.0       471
 0.0     31332
 1.0      3253
 2.0      1795
 3.0      2455
 4.0      1943
 5.0      8783
 6.0     10054
 7.0      3048
 8.0     86318
 9.0      2308
 10.0     1818
 11.0     2449
 12.0    85651
 15.0     3072
Name: count, dtype: int64

hs_highest_grade_offered value counts (public):
hs_highest_grade_offered
-2.0       2323
-1.0       8739
 0.0       3747
 1.0       2740
 2.0       9389
 3.0       7002
 4.0      22656
 5.0     179035
 6.0      51686
 7.0       2938
 8.0     181540
 9.0       5220
 10.0      1643
 11.0      2037
 12.0    298639
 13.0      1346
 14.0       191
 15.0      1383
 NaN      21571
Name: count, dtype: int64


In [9]:
# Fill rate summary by school_type
pub_mask  = df['school_type'].astype('string') == 'public'
priv_mask = df['school_type'].astype('string') == 'private'

hs_cols = [c for c in df.columns if c.startswith('hs_')]

rows = []
for col in hs_cols:
    pub_fill  = df.loc[pub_mask,  col].notna().mean() * 100
    priv_fill = df.loc[priv_mask, col].notna().mean() * 100

    if pub_fill > 0 and priv_fill > 0:
        coverage = 'both'
    elif pub_fill > 0:
        coverage = 'public only'
    else:
        coverage = 'private only'

    rows.append({
        'column':        col,
        'coverage':      coverage,
        'public_%_fill': round(pub_fill,  1),
        'private_%_fill': round(priv_fill, 1),
    })

summary = pd.DataFrame(rows).set_index('column')
print(summary.to_string())

                                        coverage  public_%_fill  private_%_fill
column                                                                         
hs_id                                       both          100.0           100.0
hs_name                                     both           98.7           100.0
hs_city                                     both           97.3           100.0
hs_state                                    both           97.3           100.0
hs_zip                                      both           97.3           100.0
hs_ctyname                          private only            0.0           100.0
hs_cty_fips                                 both           97.3           100.0
hs_lat                                      both           97.3           100.0
hs_long                                     both           97.3           100.0
hs_pct_white                         public only           94.5             0.0
hs_pct_black                         pub

In [10]:
# Fill private school enrollment, race %, and students-per-teacher from PSS 2019-20
pss_demo = pd.read_csv(DATA_DIR / 'pss_2019-20.csv', low_memory=False, usecols=[
    'PPIN', 'P305', 'P330', 'P325', 'P320', 'P316', 'P310', 'P318', 'P332', 'P385'
])
pss_demo['PPIN'] = pss_demo['PPIN'].astype('string').str.strip()

# Coerce all demo columns to numeric
demo_cols = ['P305', 'P330', 'P325', 'P320', 'P316', 'P310', 'P318', 'P332', 'P385']
for c in demo_cols:
    pss_demo[c] = pd.to_numeric(pss_demo[c], errors='coerce')

# Derived columns
pss_demo['_enrollment']          = pss_demo['P305']
pss_demo['_pct_white']           = pss_demo['P330'] / pss_demo['P305']
pss_demo['_pct_black']           = pss_demo['P325'] / pss_demo['P305']
pss_demo['_pct_hispanic']        = pss_demo['P320'] / pss_demo['P305']
pss_demo['_pct_asian']           = pss_demo['P316'] / pss_demo['P305']
pss_demo['_pct_aian']            = pss_demo['P310'] / pss_demo['P305']
pss_demo['_pct_nhpi']            = pss_demo['P318'] / pss_demo['P305']
pss_demo['_pct_two_or_more']     = pss_demo['P332'] / pss_demo['P305']
pss_demo['_students_per_teacher'] = pss_demo['P305'] / pss_demo['P385']

# Zero enrollment -> NaN for rates
zero_enroll = pss_demo['P305'].isna() | (pss_demo['P305'] == 0)
rate_cols = ['_pct_white', '_pct_black', '_pct_hispanic', '_pct_asian',
             '_pct_aian', '_pct_nhpi', '_pct_two_or_more']
pss_demo.loc[zero_enroll, rate_cols] = pd.NA

# Zero teachers -> NaN for ratio
zero_teach = pss_demo['P385'].isna() | (pss_demo['P385'] == 0)
pss_demo.loc[zero_teach, '_students_per_teacher'] = pd.NA

pss_demo_dedup = pss_demo.drop_duplicates(subset=['PPIN'])

# Column mapping: PSS derived -> final_data hs_ col
fill_map = {
    '_enrollment':           'hs_enrollment',
    '_pct_white':            'hs_pct_white',
    '_pct_black':            'hs_pct_black',
    '_pct_hispanic':         'hs_pct_hispanic',
    '_pct_asian':            'hs_pct_asian',
    '_pct_aian':             'hs_pct_aian',
    '_pct_nhpi':             'hs_pct_nhpi',
    '_pct_two_or_more':      'hs_pct_two_or_more',
    '_students_per_teacher': 'hs_students_per_teacher',
}

priv_mask = df['school_type'].astype('string') == 'private'

merged_demo = (
    df.loc[priv_mask, ['hs_id']]
      .merge(pss_demo_dedup[['PPIN'] + list(fill_map.keys())],
             left_on='hs_id', right_on='PPIN', how='left')
      .set_index(df.loc[priv_mask].index)
)

for pss_col, hs_col in fill_map.items():
    missing = priv_mask & df[hs_col].isna()
    df.loc[missing, hs_col] = merged_demo.loc[missing, pss_col]
    print(f'  {hs_col}: filled {missing.sum():,} rows')

print('\nMissing % for private rows after fill:')
print((df.loc[priv_mask, list(fill_map.values())].isna().mean() * 100).round(2))

  hs_enrollment: filled 244,750 rows
  hs_pct_white: filled 244,750 rows
  hs_pct_black: filled 244,750 rows
  hs_pct_hispanic: filled 244,750 rows
  hs_pct_asian: filled 244,750 rows
  hs_pct_aian: filled 244,750 rows
  hs_pct_nhpi: filled 244,750 rows
  hs_pct_two_or_more: filled 244,750 rows
  hs_students_per_teacher: filled 244,750 rows

Missing % for private rows after fill:
hs_enrollment              0.00
hs_pct_white               0.00
hs_pct_black               0.01
hs_pct_hispanic            0.00
hs_pct_asian               0.00
hs_pct_aian                0.00
hs_pct_nhpi                0.00
hs_pct_two_or_more         0.00
hs_students_per_teacher    2.00
dtype: float64


In [12]:
# Mark public-only columns as -10 for all private rows (not applicable)
public_only_cols = [
    'hs_title_i_status',
    'hs_urban_centric_locale',
    'hs_charter',
    'hs_magnet',
    'hs_pct_free_or_reduced_price_lunch',
]
priv_mask = df['school_type'].astype('string') == 'private'
df.loc[priv_mask, public_only_cols] = -10
print('Set -10 (not applicable) for private rows on:', public_only_cols)

Set -10 (not applicable) for private rows on: ['hs_title_i_status', 'hs_urban_centric_locale', 'hs_charter', 'hs_magnet', 'hs_pct_free_or_reduced_price_lunch']


In [13]:
out_path = DATA_DIR / 'final_data_v2.csv'
df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
df.head()

Saved: /Users/bryceclement/Desktop/c2i/data/final_data_v2.csv


,college_id,cycle,school_type,hs_id,hs_name,col_name,col_city,col_st,col_zip,col_type,...,hs_pct_two_or_more,hs_title_i_status,hs_urban_centric_locale,hs_charter,hs_magnet,hs_enrollment,hs_pct_free_or_reduced_price_lunch,hs_students_per_teacher,hs_school_level,hs_highest_grade_offered
0,45896401,2022,public,450201000242,Gregg Middle,STRAYER UNIVERSITY - CHARLESTON CAMPUS,NORTH CHARLESTON,SC,29418.0,3.0,...,0.088235,NaN,21.0,0.0,NaN,850.0,0.757647,14.655172,middle,8.0
1,101189,2023,private,A1900014,EASTWOOD CHRISTIAN SCHOOL,FAULKNER UNIVERSITY,MONTGOMERY,AL,36109.0,2.0,...,0.032389,-10.0,-10.0,-10.0,-10.0,247.0,-10.000000,12.350000,combined,12.0
2,46114802,2020,public,360007705522,HARVEY MILK HIGH SCHOOL,NEW YORK FILM ACADEMY - NEW YORK CAMPUS,NEW YORK,NY,10004.0,3.0,...,0.000000,5.0,11.0,0.0,0.0,58.0,0.896552,3.866667,high,12.0
3,200800,2022,public,390434800040,Leggett Community Learning Center,UNIVERSITY OF AKRON MAIN CAMPUS,AKRON,OH,44325.0,1.0,...,0.086253,NaN,12.0,0.0,NaN,371.0,NaN,12.793103,elementary,5.0
4,155089,2020,public,201113000926,Riverton High,FRIENDS UNIVERSITY,WICHITA,KS,67213.0,2.0,...,0.073733,4.0,31.0,0.0,0.0,217.0,0.460829,14.466667,high,12.0
